# geeViz Esri Integration Examples

Demonstrates `geeViz.esriLib` — searching any ArcGIS Portal, adding Esri
Image Services, Feature Services, and Map Services alongside Google Earth
Engine layers in the geeViz viewer.

| Section | What it covers |
|---|---|
| 1. Portal search | Search IIPP, AGOL, and custom portals for datasets |
| 2. Service metadata | Inspect any Esri service endpoint |
| 3. Image Service | Add an Esri ImageServer as XYZ tiles |
| 4. Feature Service | Fetch and add vector GeoJSON |
| 5. Combined view | Esri layers + Earth Engine layers together |
| 6. Auto-dispatch | `addEsriService` for unknown service types |
| 7. Secured portals | Token-based auth pattern |

**No API key required** — IIPP and most agency portals are public.
Token-gated examples show the pattern but are clearly marked.

[![github](https://img.shields.io/badge/-see%20sources-white?logo=github&labelColor=555)](https://github.com/gee-community/geeviz/blob/master/examples/esri_integration.ipynb) 
[![github](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gee-community/geeViz/blob/master/examples/esri_integration.ipynb)

---
## Setup

In [1]:
import geeViz.esriLib as el
# Search and metadata come from georest, geeViz's REST client;
# esriLib puts the services on a geeViz map.
from georest.esri import portal
import geeViz.geeView as gv

Map = gv.Map
ee = gv.ee

print("Available portals:", list(portal.PORTALS.keys()))
print("IIPP URL:", portal.PORTALS['iipp'])

geeViz: Earth Engine ready (project='geeviz-geo-agent', source=eeauth-proxy)
Available portals: ['iipp', 'agol', 'usgs', 'noaa', 'usfs', 'nasa']
IIPP URL: https://imagery.geoplatform.gov/iipp


[geeViz.eeAuth] EE initialized via proxy: http://127.0.0.1:1234/ee-api (attached to existing detached proxy, tenant_header=X-geeViz-Creds)


---
## 1. Portal Search

`searchPortal` hits the standard ArcGIS `/sharing/rest/search` endpoint —
identical across IIPP, ArcGIS Online, and any Enterprise install.

In [2]:
# Search IIPP (default) for NAIP imagery
results = portal.searchPortal("naip imagery", limit=10)
print(f"Found {len(results)} results on IIPP:\n")
for r in results:
    print(f"  [{r['type']}] {r['title']}")
    print(f"    URL: {r['url']}")

Found 10 results on IIPP:

  [Image Service] NAIP2023_CONUS
    URL: https://imagery.geoplatform.gov/iipp/rest/services/NAIP/NAIP2023_CONUS/ImageServer
  [Image Service] NAIP_plus
    URL: https://imagery.geoplatform.gov/iipp/rest/services/NAIP/NAIP_plus/ImageServer
  [Image Service] NAIP2022_CONUS
    URL: https://imagery.geoplatform.gov/iipp/rest/services/NAIP/NAIP2022_CONUS/ImageServer
  [Image Service] NAIP2021_CONUS
    URL: https://imagery.geoplatform.gov/iipp/rest/services/NAIP/NAIP2021_CONUS/ImageServer
  [Map Service] NAIP2023_CONUS_AcquisitionDates
    URL: https://imagery.geoplatform.gov/iipp/rest/services/NAIP/NAIP2023_CONUS_AcquisitionDates/MapServer
  [Image Service] NAIP2019_CONUS
    URL: https://imagery.geoplatform.gov/iipp/rest/services/NAIP/NAIP2019_CONUS/ImageServer
  [Map Service] NAIP2022_CONUS_AcquisitionDates
    URL: https://imagery.geoplatform.gov/iipp/rest/services/NAIP/NAIP2022_CONUS_AcquisitionDates/MapServer
  [Image Service] NAIP2018_CONUS
    URL: https:

In [3]:
# Search ArcGIS Online for wildfire perimeter data
agol_results = portal.searchPortal("wildfire perimeter 2023", portal="agol", limit=5)
print(f"Found {len(agol_results)} results on AGOL:")
for r in agol_results:
    print(f"  [{r['type']}] {r['title']}")

Found 5 results on AGOL:
  [Feature Service] Wildfire Aware Incident, Population, and Environment Enriched Layer 
  [Feature Service] WFIGS - Wildland Fire Perimeters 2023
  [Feature Service] California Wildfire Perimeters and Incident Points
  [Feature Service] Historic Wildfire Perimeters - Siskiyou County Only (CAL FIRE)
  [Feature Service] Rico Fire (2023) Perimeter - HFD


In [4]:
# Search with data_only=False to see all item types
all_types = portal.searchPortal("imagery", portal="agol", limit=5, data_only=False)
print("All item types (no filter):")
for r in all_types:
    print(f"  [{r['type']}] {r['title']}")

All item types (no filter):
  [Web Map] Imagery Hybrid
  [Oriented Imagery Catalog] Mapillary Oriented Imagery Catalog
  [Map Service] World Imagery (WGS84)
  [Map Service] World Imagery
  [Web Map] Imagery with Metadata


In [5]:
# Power-user raw query — portal DSL syntax, bypasses data_only entirely.
# orgid:T4QMspbfLg3qTGWY is the National Interagency Fire Center organization.
raw = portal.searchPortal(
    "",
    portal="agol",
    raw_q='type:"Feature Service" tags:fire orgid:T4QMspbfLg3qTGWY',
    limit=5,
)
print(f"Raw DSL results: {len(raw)}")
for r in raw:
    print(f"  {r['title']}")

Raw DSL results: 5
  Historic Perimeters 2019
  National Dispatch Boundaries Public
  AK Fire Management Options
  National GACC Boundaries Public
  AICC Dashboards Data


---
## 2. Service Metadata

Inspect any service endpoint — useful before adding it to the map.

In [6]:
# Pick the first Image Service result from the IIPP search above
image_results = [r for r in results if r['type'] == 'Image Service']
# NAIP_plus is the newest imagery for every state. The per-year services
# (NAIP2023_CONUS, ...) cover only the states flown that year -- 2023 has
# no Utah -- so they draw nothing over Salt Lake City.
naip = next((r for r in image_results if r['title'] == 'NAIP_plus'),
            image_results[0] if image_results else None)

if naip:
    svc_url = naip['url']
    print(f"Inspecting: {svc_url}")
    meta = portal.getServiceMetadata(svc_url)
    print(f"  Name       : {meta.get('name')}")
    print(f"  Band count : {meta.get('bandCount')}")
    print(f"  Pixel type : {meta.get('pixelType')}")
    print(f"  Min scale  : {meta.get('minScale')}")
    print(f"  Max scale  : {meta.get('maxScale')}")
    extent = meta.get('extent', {})
    print(f"  Extent     : {extent}")
else:
    print("No Image Service results found — try a broader search query.")

Inspecting: https://imagery.geoplatform.gov/iipp/rest/services/NAIP/NAIP_plus/ImageServer
  Name       : NAIP/NAIP_plus
  Band count : 4
  Pixel type : U8
  Min scale  : 0
  Max scale  : 0
  Extent     : {'ymin': 2022493.1094999975, 'xmin': -17846192.4, 'ymax': 6350184.509499997, 'xmax': -7291004.999999998, 'spatialReference': {'latestWkid': 3857, 'wkid': 102100}}


In [7]:
# Inspect a public Feature Service (NIFC fire perimeters — always public)
nifc_url = ("https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/"
            "WFIGS_Interagency_Perimeters/FeatureServer/0")
try:
    nifc_meta = portal.getServiceMetadata(nifc_url)
    print(f"Service name : {nifc_meta.get('name')}")
    print(f"Geometry type: {nifc_meta.get('geometryType')}")
    fields = [f['name'] for f in nifc_meta.get('fields', [])[:8]]
    print(f"Fields (first 8): {fields}")
except ConnectionError as e:
    print(f"Network error: {e}")

Service name : Perimeters
Geometry type: esriGeometryPolygon
Fields (first 8): ['OBJECTID', 'poly_SourceOID', 'poly_IncidentName', 'poly_FeatureCategory', 'poly_MapMethod', 'poly_GISAcres', 'poly_DeleteThis', 'poly_FeatureAccess']


---
## 3. Adding an Esri Image Service

A **cached** Image Service is drawn as XYZ tiles from
`<service_url>/tile/{z}/{y}/{x}` (ArcGIS order: y before x).
Most are **not** cached -- including every NAIP service on IIPP -- and
their `/tile` URLs answer 404. `addEsriImageService` reads the service
metadata and draws those through `exportImage` instead, re-rendered for
the view on each pan and zoom.


In [8]:
Map.clearMap()

# ESRI World Imagery basemap (always public, always cached)
el.addEsriMapService(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer",
    name="ESRI World Imagery",
    viz_params={"opacity": 0.7},
)

# Add the NAIP ImageServer found in the search above
if naip:
    el.addEsriImageService(
        naip,   # pass the full search result dict
        name=naip['title'],
        viz_params={"opacity": 0.85},
    )

Map.centerObject(ee.Geometry.Point([-111.89, 40.77]), 10)
Map.view()

Adding Esri Image Service: ESRI World Imagery


Adding Esri Image Service (dynamic, exportImage): NAIP_plus


Starting webmap
Using eeCreds proxy at http://127.0.0.1:1234/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1234/geoview/geoview.html?v=1795650407658&doc=../geeView/src/gee/gee-run/map-a323b232bac11e73-51b8495b861d.json


---
## 4. Adding an Esri Feature Service

Feature Services are fetched as GeoJSON and rendered as vector layers.

**Always does a pre-flight count check** before downloading features.
If the count exceeds `max_features`, a `ValueError` is raised with
remediation options (adjust `where`, raise `max_features`, or paginate).

In [9]:
Map.clearMap()

# NIFC interagency fire perimeters -- large recent fires.
# The service generalizes the geometry to ~5 m before sending it
# (simplify="auto", the default): about 6 MB in ~6 s, where the
# full-resolution perimeters are 34 MB and ~30 s. Pass simplify=False
# for exact geometry.
nifc_url = ("https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/"
            "WFIGS_Interagency_Perimeters/FeatureServer/0")

try:
    el.addEsriFeatureService(
        nifc_url,
        name="NIFC Fire Perimeters (>100k acres, 2023+)",
        where="attr_IncidentSize > 100000 AND attr_FireDiscoveryDateTime >= DATE '2023-01-01'",
        max_features=500,
        viz_params={
            "color": "FF4500",
            "opacity": 0.7,
            "strokeWidth": 2,
        },
    )
except ValueError as e:
    print(f"Feature count exceeded: {e}")
except ConnectionError as e:
    print(f"Network error: {e}")

Map.centerObject(ee.Geometry.BBox(-125, 31, -100, 50), 5)
Map.view()

Adding Esri Feature Service: NIFC Fire Perimeters (>100k acres, 2023+) (38 features)
Adding layer: NIFC Fire Perimeters (>100k acres, 2023+)
Starting webmap


Using eeCreds proxy at http://127.0.0.1:1234/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1234/geoview/geoview.html?v=1795650412027&doc=../geeView/src/gee/gee-run/map-a323b232bac11e73-73667e61643d.json


In [10]:
# Demonstrate the overflow ValueError
print("Attempting to fetch all perimeters with max_features=10 (will fail on purpose)...")
try:
    el.addEsriFeatureService(
        nifc_url,
        max_features=10,
    )
except ValueError as e:
    print(f"ValueError raised as expected:\n{e}")

Attempting to fetch all perimeters with max_features=10 (will fail on purpose)...


ValueError raised as expected:
Service layer has 43,389 matching features (max_features=10).
Increase max_features OR pass a `where` clause to filter, e.g. where="STATE_FIPS='06'", or pass if_more="page" to fetch them all a page at a time.


---
## 5. Combined View — Esri + Earth Engine

Esri layers and GEE layers can be combined freely.  The viewer handles
both `tileMapService` (Esri) and standard GEE serialized layers.

In [11]:
Map.clearMap()

# --- Earth Engine layer: MODIS burned area ---
burned = (
    ee.ImageCollection("MODIS/061/MCD64A1")
    .filterDate("2020-01-01", "2023-12-31")
    .select("BurnDate")
    .max()
    .selfMask()
)
Map.addLayer(
    burned,
    {"min": 1, "max": 366, "palette": ["yellow", "orange", "red", "darkred"]},
    "MODIS Burned Area 2020-2023",
    visible=True,
)

# --- Esri layer: ESRI World Imagery basemap ---
el.addEsriMapService(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer",
    name="ESRI World Imagery (basemap)",
    viz_params={"opacity": 0.5},
)

# --- Esri Feature Service: fire perimeters as vector overlay ---
try:
    el.addEsriFeatureService(
        nifc_url,
        name="NIFC Perimeters (vector)",
        where=("attr_IncidentSize > 200000"
               " AND attr_FireDiscoveryDateTime >= DATE '2020-01-01'"
               " AND attr_FireDiscoveryDateTime < DATE '2024-01-01'"),
        max_features=200,
        viz_params={"color": "FF6600", "strokeWidth": 2, "opacity": 0.6},
    )
except (ValueError, ConnectionError) as e:
    print(f"Feature service unavailable: {e}")

Map.centerObject(ee.Geometry.BBox(-125, 31, -100, 50), 5)
Map.view()

Adding layer: MODIS Burned Area 2020-2023
Adding Esri Image Service: ESRI World Imagery (basemap)


Adding Esri Feature Service: NIFC Perimeters (vector) (18 features)
Adding layer: NIFC Perimeters (vector)


Starting webmap


Using eeCreds proxy at http://127.0.0.1:1234/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1234/geoview/geoview.html?v=1795650415583&doc=../geeView/src/gee/gee-run/map-a323b232bac11e73-f23c6310a745.json


---
## 6. Auto-dispatch with `addEsriService`

`addEsriService` inspects the URL and calls the right typed helper
automatically.  Useful when looping over mixed search results.

In [12]:
Map.clearMap()

# Mixed search — whatever types come back, auto-dispatch handles them
mixed = portal.searchPortal("land cover", portal="agol", limit=5)
print(f"Search returned {len(mixed)} items:")

for r in mixed:
    url = r.get('url') or ''
    print(f"  [{r['type']}] {r['title']} — {url[:60] if url else '(no URL)'}...")
    if not url:
        print("    (no service URL, skipping)")
        continue
    try:
        el.addEsriService(
            r,
            name=r['title'],
            max_features=500,
        )
    except ValueError as e:
        print(f"    Skipped: {e}")
    except Exception as e:
        print(f"    Error: {type(e).__name__}: {e}")

Map.centerObject(ee.Geometry.BBox(-125, 24, -66, 50), 4)
Map.view()

Search returned 5 items:
  [Image Service] Sentinel-2 10m Land Use/Land Cover Time Series — https://ic.imagery1.arcgis.com/arcgis/rest/services/Sentinel...


Adding Esri Image Service (dynamic, exportImage): Sentinel-2 10m Land Use/Land Cover Time Series
  [Image Service] USA Annual NLCD - Land Cover — https://di-nlcd.img.arcgis.com/arcgis/rest/services/USA_NLCD...


Adding Esri Image Service (dynamic, exportImage): USA Annual NLCD - Land Cover
  [Image Service] USA Legacy NLCD Land Cover — https://landscape10.arcgis.com/arcgis/rest/services/USA_NLCD...


Adding Esri Image Service: USA Legacy NLCD Land Cover
  [Image Service] Global Land Cover 1992-2020 — https://earthobs3.arcgis.com/arcgis/rest/services/ESA_CCI_La...


Adding Esri Image Service: Global Land Cover 1992-2020
  [Deep Learning Package] Land Cover Classification (Sentinel-2) — (no URL)...
    (no service URL, skipping)


Starting webmap
Using eeCreds proxy at http://127.0.0.1:1234/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:1234/geoview/geoview.html?v=1795650417802&doc=../geeView/src/gee/gee-run/map-a323b232bac11e73-71b7814acc9e.json


---
## 7. Secured Portals — Token Auth

Most public portals (IIPP, AGOL public items) need no token.  For
secured enterprise portals, generate a token first.

```
POST https://your-portal.gov/portal/sharing/rest/generateToken
  username=myuser
  password=mypassword
  client=requestip
  expiration=60
  f=json
```

The response contains `{"token": "...", "expires": ...}`.

In [13]:
import urllib.parse
import urllib.request
import json

def get_esri_token(portal_url: str, username: str, password: str,
                   expiration: int = 60) -> str:
    """Obtain a short-lived ArcGIS token for a secured portal.

    Args:
        portal_url: Base URL of the portal (e.g. 'https://gis.myagency.gov/portal').
        username: Portal username.
        password: Portal password.
        expiration: Token lifetime in minutes (default 60).

    Returns:
        Token string.
    """
    token_url = f"{portal_url.rstrip('/')}/sharing/rest/generateToken"
    payload = urllib.parse.urlencode({
        "username": username,
        "password": password,
        "client": "requestip",
        "expiration": str(expiration),
        "f": "json",
    }).encode("utf-8")
    req = urllib.request.Request(token_url, data=payload,
                                 headers={"User-Agent": "geeViz/esriLib"})
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    if "error" in data:
        raise RuntimeError(f"Token error: {data['error']}")
    return data["token"]


# ---- EXAMPLE (replace with real credentials for a secured portal) ----
# token = get_esri_token(
#     "https://gis.myagency.gov/portal",
#     username="myuser",
#     password="mypassword",
# )
# portal.searchPortal("classified data",
#                 portal="https://gis.myagency.gov/portal",
#                 token=token)
# el.addEsriFeatureService(
#     "https://gis.myagency.gov/arcgis/rest/services/Secure/FeatureServer/0",
#     token=token,
# )
print("Token helper defined. Uncomment the example block and fill in credentials.")

Token helper defined. Uncomment the example block and fill in credentials.


---
## Module Reference

| Function | Description |
|---|---|
| `searchPortal(query, portal, ...)` | Search any ArcGIS Portal |
| `getServiceMetadata(url, token)` | Fetch `?f=json` metadata for any service |
| `addEsriImageService(url_or_result, ...)` | Add ImageServer as XYZ tiles |
| `addEsriMapService(url_or_result, ...)` | Add cached MapServer as XYZ tiles |
| `addEsriFeatureService(url_or_result, ...)` | Fetch + add vector GeoJSON layer |
| `addEsriService(url_or_result, ...)` | Auto-detect service type, then dispatch |

| Constant | Description |
|---|---|
| `PORTALS` | Dict of portal short names → base URLs |

### Key pitfalls

- **ArcGIS tile order** is `{z}/{y}/{x}`, not `{z}/{x}/{y}`.  The
  `addEsriImageService` / `addEsriMapService` helpers emit the correct order.
- **Feature Service overflow** raises `ValueError` by design — silent
  truncation would produce partial-data analysis results.
- **outSR=4326** is always requested on Feature Service queries so the
  viewer renders GeoJSON in WGS84 natively without re-projection.
- **Token expiry** — ArcGIS tokens default to 60-minute lifetimes.
  Re-generate before long notebook sessions.
- **CORS** — public Esri endpoints (AGOL, IIPP) send permissive CORS
  headers.  Private Enterprise installs behind a firewall need a
  backend proxy.